# The DSI Denodo Schema — Tables, Fields, and Relationships



This notebook answers three questions in order:

1. **What tables do we have?**
2. **What fields are in each table?**
3. **How are the tables linked together?**

It closes with a worked example tracing one real view through every table.

## 1. The big picture — one conceptual model, four physical tables

The approved **tri-layer model** describes the catalog conceptually; the physical schema realizes it as **four tables**:

```
 CONCEPTUAL (tri-layer)                 PHYSICAL (v1.0 schema)
 ─────────────────────                  ──────────────────────
 Layer 1   VDB / database        ──▶    denodo_databases
 Layer 2   Views (datasets)      ──▶    denodo_views
 Layer 2.5 Columns (schema)      ──▶    denodo_columns
 Layer 3   Additional resources  ──▶    folded INTO denodo_views
                                        (3 nullable columns — Decision Point 1, Option A)
 Custom properties (341+)        ──▶    denodo_properties  (long EAV — Decision Point 2)
```

Two structural facts drive everything below:

- **View identity in Denodo is inherently the pair `(view_name, db_name)`** — the `view-details` API requires both parameters, Denodo namespaces views per VDB, and a second VDB (`ops_core_publication`) has now been observed empirically alongside `dataportal`. So the composite key appears in every table.
- **Layer 3 is folded, not deleted.** `documentation_url` proved strictly 1:1 with views (544/544), so a separate resources table would add a join without adding expressiveness. The tri-layer model is intact conceptually; folding is a physical-representation choice, reversible additively if 1:many attachments ever appear.

## 2. Table by table

### 2.1 `denodo_databases` — Layer 1 (one row per VDB)

#### the post-migration dev catalog now lists 4,418 views in total — 4,414 in dataportal, plus 4 views in a second database called `"ops_core_publication"`

| Field | Type | Nullable | Meaning / source |
|---|---|---|---|
| `db_name` | str | No | **Key.** Distinct `db` / `databaseName` values from the list endpoint (e.g. `"dataportal"`, `"ops_core_publication"`) |
| `description` | str | Yes | Database-level description (source verification pending — Section 5.4 open item; `None` acceptable for v1.0) |
| `server_id` | int | No | The `serverId` request parameter the client sends (currently `1`) |
| `view_count` | int | No | Denormalized count of ingested views in this VDB, refreshed on each fetch |
| `fetched_at` | str | No | Provenance: when this snapshot was taken |
| `source_env` | str | No | Provenance: `"dev"` or `"prod"` |

Trivial today (two rows), designed to hold many rows as additional VDBs onboard — no code path hard-codes `dataportal`.

### 2.2 `denodo_views` — Layer 2, with Layer 3 folded in (one row per view)

| Field | Type | Nullable | Meaning / source |
|---|---|---|---|
| `view_name` | str | No | **Key part 1.** From list endpoint `name` |
| `db_name` | str | No | **Key part 2.** From list endpoint `db`/`databaseName` (dev/prod name drift normalized) |
| `description` | str | Yes | From `view-details`; HTML-stripped (§3.2) |
| `source_system` | str | Yes | *Folded Layer 3.* Reclassified from the `ODS DB Link` property — Oracle db-link lineage (e.g. `EBS_LINK`); literal `"NOLINK"` → `None` |
| `access_instructions` | str | Yes | *Folded Layer 3.* Reclassified from `Access Role Request` — ~93% are `accessit.lanl.gov` URLs |
| `documentation_url` | str | Yes | *Folded Layer 3.* URL extracted from `href` inside the description **before** HTML stripping (§3.3); scalar by the 544/544 1:1 finding |
| `fetch_status` | str | No | `"ok"` or `"error"` — failed fetches keep their row (§4.1), so failures are queryable, never hidden |
| `fetched_at` | str | No | Provenance |
| `source_env` | str | No | Provenance |

### 2.3 `denodo_columns` — Layer 2.5 (one row per column of a view)

| Field | Type | Nullable | Meaning / source |
|---|---|---|---|
| `view_name` | str | No | **Key part 1** (FK → `denodo_views`) |
| `db_name` | str | No | **Key part 2** (FK → `denodo_views`) |
| `column_name` | str | No | **Key part 3** |
| `ordinal_position` | int | No | 1-based order of the column within the view |
| `data_type` | str | No | Denodo-reported type |
| `description` | str | Yes | Column description, HTML-stripped |

Sourced solely from `view-details` — Maxen confirmed this endpoint is authoritative for column schema. Not a layer of the tri-layer model itself: it is nested *inside* Layer 2 (it describes what columns a view has).

### 2.4 `denodo_properties` — custom properties as long EAV (one row per property value)

| Field | Type | Nullable | Meaning / source |
|---|---|---|---|
| `view_name` | str | No | **Key part 1** (FK → `denodo_views`) |
| `db_name` | str | No | **Key part 2** (FK → `denodo_views`) |
| `property_name` | str | No | **Key part 3.** Group-qualified: `groupName + "/" + propertyName` (e.g. `Details/Data Classification`, `Additional Information/More Help for Web Services`) — near-duplicates like `Details/Access Role Request` vs `Details/Access Request` are never merged |
| `property_value` | str | Yes | Normalized per §3 (HTML strip; href-first URL extraction) |

Why long EAV instead of 341 wide columns: the property set is sparse and **will drift** — PROD already stages 36 DCAT groups with no data yet; when that rollout starts, EAV absorbs it with **zero schema migrations**. Extraction reads the union of all three property maps in `view-details` (`summaryPropertyMap` + `generalTabPropertyMap` + `customTabPropertyMap`), keyed on `propertyName`. Two properties are *routed out* of this table into `denodo_views` columns (`source_system`, `access_instructions`) so the same fact never lives in two places.

## 3. How the tables link together

```
                    ┌──────────────────────┐
                    │   denodo_databases   │   1 row per VDB
                    │  PK: (db_name)       │
                    └──────────┬───────────┘
                               │ 1 : N     join on  db_name
                               ▼
                    ┌──────────────────────┐
                    │     denodo_views     │   1 row per view
                    │  PK: (view_name,     │   (Layer 3 folded in as
                    │       db_name)       │    3 nullable columns)
                    └────┬────────────┬────┘
              1 : N      │            │      1 : N
   join on (view_name,   │            │   join on (view_name,
            db_name)     ▼            ▼            db_name)
        ┌──────────────────────┐  ┌──────────────────────┐
        │    denodo_columns    │  │  denodo_properties   │
        │ PK: (view_name,      │  │ PK: (view_name,      │
        │      db_name,        │  │      db_name,        │
        │      column_name)    │  │      property_name)  │
        └──────────────────────┘  └──────────────────────┘
```

Reading the diagram:

- **`denodo_databases` → `denodo_views`** — 1:N on `db_name`. One database holds many views; `view_count` on the database row is the denormalized size of that fan-out.
- **`denodo_views` → `denodo_columns`** — 1:N on `(view_name, db_name)`. Each view's schema, one row per column, ordered by `ordinal_position`.
- **`denodo_views` → `denodo_properties`** — 1:N on `(view_name, db_name)`. Each view's custom metadata, one row per populated property.
- Every child key **extends** the parent key (`+ column_name`, `+ property_name`) — so any child row can always be traced back to exactly one view, and any view to exactly one database. Key uniqueness is asserted by `contract_validator.py`, never assumed.

## 4. Worked example — one real view through all four tables

View `admin_option_type_fvts` in `dataportal` (the `custom_tab` golden fixture):

**`denodo_databases`** — its parent database:

| db_name | description | server_id | view_count | fetched_at | source_env |
|---|---|---|---|---|---|
| dataportal | None | 1 | 1 | 2026-07-28T… | dev |

**`denodo_views`** — the view row (folded Layer 3 in bold):

| field | value |
|---|---|
| view_name | admin_option_type_fvts |
| db_name | dataportal |
| description | (HTML-stripped text) |
| **source_system** | None |
| **access_instructions** | https://accessit.lanl.gov |
| **documentation_url** | None |
| fetch_status | ok |

**`denodo_columns`** — one row per column of the view (`view_name`, `db_name` repeat on every row; `ordinal_position` 1, 2, 3, …).

**`denodo_properties`** — its populated custom properties, including the one that proves the three-map union works:

| property_name | property_value |
|---|---|
| Details/Data Classification | … |
| Additional Information/More Help for Web Services | https://collaborate.lanl.gov/x/… |

That last row comes from `customTabPropertyMap` — a value that lives on the *Additional Properties* tab in the Denodo UI, captured by the same single `view-details` call as everything else.

## 5. Status

- Contract: `DATA_CONTRACT_v10.md`, frozen pending sign-off on the two decision points (DP1 settled as Option A; DP2 recommends long EAV).
- Evidence: 8/8 golden fixtures — including an error-path fixture and a second-VDB fixture — validate **green** against this schema.
- Next: with sign-off, Step 2/3 builds `denodo.py` against this exact shape, inheriting the fixtures + validator as its test suite.

---
*Optional: run the cell below **next to the repo** (`expected/` + `contract_validator.py` present) to re-verify the green status live. It degrades gracefully if the files are not present.*

In [ ]:
import glob, os, subprocess, sys

if os.path.exists("contract_validator.py") and glob.glob("expected/*.py"):
    fails = []
    for f in sorted(glob.glob("expected/*.py")):
        p = subprocess.run([sys.executable, f], capture_output=True, text=True)
        ok = "OK" in p.stdout
        print(f"{os.path.basename(f):<22} {'OK' if ok else 'FAIL'}")
        if not ok:
            fails.append(f)
    print("\nALL GREEN" if not fails else f"FAILING: {fails}")
else:
    print("(expected/ or contract_validator.py not found here -- "
          "run from the repo directory to re-verify live)")